<div style="background: linear-gradient(135deg, #1a1a2e, #16213e, #0f3460); padding: 30px; border-radius: 15px; text-align:center;">
  <h1 style="color:#e94560; font-size:2.5em; margin:0;">🎬 Movie Genre Classifier</h1>
  <h3 style="color:#a8b2d8; margin-top:10px;">EfficientNetB3 — Multi-Label Classification</h3>
  <p style="color:#64ffda;">Fixed & Optimized Version ✅</p>
</div>

---
## 📦 الخطوة صفر: تثبيت المكتبات

In [ ]:
!pip install tensorflow scikit-learn pandas numpy matplotlib pillow tqdm -q

---
## 📁 الخطوة الأولى: Import المكتبات

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm import tqdm

print(f"✅ TensorFlow version: {tf.__version__}")
print(f"✅ GPU available: {len(tf.config.list_physical_devices('GPU'))} GPU(s)")

---
## 📋 الخطوة الثانية: قراءة ومعالجة البيانات

In [ ]:
# ============================================================
# قراءة الداتا
# ============================================================
df = pd.read_csv('movies_dataset.csv')
print(f"📊 Total movies loaded: {len(df)}")
print(f"📋 Columns: {df.columns.tolist()}")
df.head(3)

In [ ]:
# ============================================================
# ✅ FIX #1: تصحيح separator من | لـ , (الفاصل الصح في الداتا)
# ============================================================
df['genres_list'] = df['genres'].apply(
    lambda x: [g.strip() for g in x.split(',')] if isinstance(x, str) and x.strip() != '' else []
)

# تأكد إن مفيش rows بقائمة فاضية
df = df[df['genres_list'].map(len) > 0].reset_index(drop=True)

print(f"✅ Movies with valid genres: {len(df)}")
print("\n🎭 Sample genres after fix:")
for i in range(5):
    print(f"  [{df['title'].iloc[i]}] → {df['genres_list'].iloc[i]}")

In [ ]:
# ============================================================
# ✅ FIX #2: تصحيح مسار الصور (Windows backslash → forward slash)
# ============================================================
df['poster_local'] = df['poster_local'].apply(
    lambda x: x.replace('\\', '/') if isinstance(x, str) else ''
)

# فلتر الصور الموجودة فعلاً على الجهاز
df_exists = df[df['poster_local'].apply(
    lambda x: os.path.exists(x) if isinstance(x, str) and x != '' else False
)].copy().reset_index(drop=True)

print(f"✅ Found {len(df_exists)} posters out of {len(df)} movies")

# لو مش لاقي صور محلية، استخدم poster_url من الإنترنت
if len(df_exists) == 0:
    print("⚠️ No local posters found. Will use poster_url column.")
    print("   Make sure you downloaded posters in the scraping step.")

In [ ]:
# ============================================================
# ✅ FIX #3: Multi-Label Binarizer (الصح لـ multi-label task)
# ============================================================
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df_exists['genres_list'])

print(f"✅ Number of genre classes: {len(mlb.classes_)}")
print(f"📌 Classes: {list(mlb.classes_)}")
print(f"🔢 Label matrix shape: {Y.shape}")

NUM_CLASSES = len(mlb.classes_)

In [ ]:
# توزيع الـ genres (عشان تشوف الـ imbalance)
genre_counts = pd.Series(Y.sum(axis=0), index=mlb.classes_).sort_values(ascending=False)

plt.figure(figsize=(14, 5))
bars = plt.bar(genre_counts.index, genre_counts.values, color=plt.cm.plasma(np.linspace(0, 1, len(genre_counts))))
plt.xticks(rotation=45, ha='right')
plt.title('📊 Genre Distribution in Dataset', fontsize=14, fontweight='bold')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

print(genre_counts)

---
## 🔀 الخطوة الثالثة: تقسيم البيانات

In [ ]:
X_paths = df_exists['poster_local'].values

X_train, X_temp, Y_train, Y_temp = train_test_split(
    X_paths, Y, test_size=0.2, random_state=42
)
X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp, Y_temp, test_size=0.5, random_state=42
)

print(f"✅ Train:      {len(X_train)} samples")
print(f"✅ Validation: {len(X_val)} samples")
print(f"✅ Test:       {len(X_test)} samples")

---
## 🖼️ الخطوة الرابعة: Data Pipeline باستخدام tf.data

In [ ]:
# ============================================================
# إعدادات
# ============================================================
IMG_SIZE    = 300          # EfficientNetB3 optimal size
BATCH_SIZE  = 32
AUTOTUNE    = tf.data.AUTOTUNE

# ============================================================
# دالة تحميل وتجهيز الصورة
# ============================================================
def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

# ============================================================
# ✅ Augmentation للتدريب فقط
# ============================================================
def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.15)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    img = tf.image.random_saturation(img, 0.85, 1.15)
    img = tf.image.random_hue(img, 0.05)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label

# ============================================================
# بناء الـ Datasets
# ============================================================
def make_dataset(paths, labels, augment_data=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (paths, tf.cast(labels, tf.float32))
    )
    ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
    if augment_data:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
        ds = ds.shuffle(buffer_size=1000)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(X_train, Y_train, augment_data=True)
val_ds   = make_dataset(X_val,   Y_val,   augment_data=False)
test_ds  = make_dataset(X_test,  Y_test,  augment_data=False)

print("✅ Datasets ready!")

# عرض عينة من الصور بعد الـ augmentation
sample_batch = next(iter(train_ds))
fig, axes = plt.subplots(2, 5, figsize=(15, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(sample_batch[0][i].numpy())
    genres_idx = np.where(sample_batch[1][i].numpy() == 1)[0]
    genres_names = [mlb.classes_[j] for j in genres_idx]
    ax.set_title('\n'.join(genres_names), fontsize=7)
    ax.axis('off')
plt.suptitle('🎬 Sample Training Posters (After Augmentation)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🧠 الخطوة الخامسة: بناء الموديل (EfficientNetB3)

In [ ]:
# ============================================================
# ✅ EfficientNetB3 بدل B0 (أقوى بفرق واضح للـ poster task)
# ============================================================
def build_model(num_classes, img_size=300):
    base_model = EfficientNetB3(
        weights='imagenet',
        include_top=False,
        input_shape=(img_size, img_size, 3)
    )
    base_model.trainable = False  # فريز في المرحلة الأولى

    inputs = tf.keras.Input(shape=(img_size, img_size, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(512, activation='relu',
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.2)(x)
    # ✅ sigmoid لأن المهمة multi-label
    outputs = layers.Dense(num_classes, activation='sigmoid')(x)

    model = models.Model(inputs, outputs)
    return model, base_model

model, base_model = build_model(NUM_CLASSES)

# ✅ Compile بـ binary_crossentropy (الصح للـ multi-label)
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[
        'binary_accuracy',
        tf.keras.metrics.AUC(multi_label=True, name='auc')
    ]
)

model.summary()

---
## 🚀 الخطوة السادسة: المرحلة الأولى — تدريب الـ Head فقط

In [ ]:
# Callbacks
callbacks_phase1 = [
    EarlyStopping(
        monitor='val_auc',
        patience=5,
        mode='max',
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        'best_model_phase1.keras',
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=1
    )
]

print("🚀 Phase 1: Training head layers (base frozen)...")
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks_phase1,
    verbose=1
)
print("✅ Phase 1 complete!")

---
## 🔥 الخطوة السابعة: المرحلة الثانية — Fine-tuning

In [ ]:
# ============================================================
# ✅ Fine-tuning: فك تجميد آخر 50 layer من الـ base model
# ============================================================
base_model.trainable = True

# فريز كل اللي قبل آخر 50 layer
for layer in base_model.layers[:-50]:
    layer.trainable = False

total_layers    = len(base_model.layers)
trainable_now   = sum(1 for l in base_model.layers if l.trainable)
print(f"📌 Total base layers:    {total_layers}")
print(f"🔓 Unfrozen for tuning: {trainable_now}")
print(f"🔒 Still frozen:        {total_layers - trainable_now}")

# Recompile بـ learning rate أصغر بكتير
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=[
        'binary_accuracy',
        tf.keras.metrics.AUC(multi_label=True, name='auc')
    ]
)

callbacks_phase2 = [
    EarlyStopping(
        monitor='val_auc',
        patience=5,
        mode='max',
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        'best_model_phase2.keras',
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=1
    )
]

print("\n🔥 Phase 2: Fine-tuning last 50 layers...")
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks_phase2,
    verbose=1
)
print("✅ Phase 2 complete!")

---
## 📈 الخطوة الثامنة: رسم منحنيات التدريب

In [ ]:
# دمج تاريخ المرحلتين
def merge_history(h1, h2):
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history[key]
    return merged

history = merge_history(history1, history2)
phase1_end = len(history1.history['loss'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0d1117')

metrics_to_plot = [
    ('binary_accuracy', 'val_binary_accuracy', 'Accuracy',   '#00d4ff', '#ff6b6b'),
    ('loss',            'val_loss',            'Loss',        '#00ff88', '#ffd93d'),
    ('auc',             'val_auc',             'AUC Score',  '#c77dff', '#ff9500'),
]

for ax, (train_key, val_key, title, tc, vc) in zip(axes, metrics_to_plot):
    epochs = range(1, len(history[train_key]) + 1)
    ax.set_facecolor('#161b22')
    ax.plot(epochs, history[train_key], color=tc, linewidth=2, label=f'Train')
    ax.plot(epochs, history[val_key],   color=vc, linewidth=2, label=f'Val', linestyle='--')
    ax.axvline(x=phase1_end, color='white', linestyle=':', alpha=0.5, label='Fine-tune start')
    ax.set_title(title, color='white', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch', color='#8b949e')
    ax.tick_params(colors='#8b949e')
    ax.spines[:].set_color('#30363d')
    ax.legend(facecolor='#161b22', labelcolor='white')
    ax.grid(alpha=0.1, color='white')

plt.suptitle('🎬 Training Curves — Phase 1 + Fine-tuning', color='white', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Plot saved as training_curves.png")

---
## 🧪 الخطوة التاسعة: تقييم الموديل على Test Set

In [ ]:
from sklearn.metrics import classification_report, hamming_loss, f1_score

# التوقع على الـ test set
print("⏳ Running predictions on test set...")
Y_pred_proba = model.predict(test_ds, verbose=1)

# اختار أفضل threshold (افتراضي 0.3 أفضل من 0.5 في multi-label)
THRESHOLD = 0.3
Y_pred = (Y_pred_proba >= THRESHOLD).astype(int)

print(f"\n📊 Evaluation Results (Threshold={THRESHOLD}):")
print(f"  Hamming Loss: {hamming_loss(Y_test, Y_pred):.4f}  (lower is better, 0=perfect)")
print(f"  F1 Micro:     {f1_score(Y_test, Y_pred, average='micro', zero_division=0):.4f}")
print(f"  F1 Macro:     {f1_score(Y_test, Y_pred, average='macro', zero_division=0):.4f}")
print(f"  F1 Weighted:  {f1_score(Y_test, Y_pred, average='weighted', zero_division=0):.4f}")

print("\n📋 Per-Genre Report:")
print(classification_report(Y_test, Y_pred, target_names=mlb.classes_, zero_division=0))

In [ ]:
# ============================================================
# إيجاد أفضل threshold لكل genre بشكل منفصل
# ============================================================
thresholds = np.arange(0.1, 0.9, 0.05)
best_f1, best_threshold = 0, 0.3

for t in thresholds:
    y_pred_t = (Y_pred_proba >= t).astype(int)
    f1 = f1_score(Y_test, y_pred_t, average='micro', zero_division=0)
    if f1 > best_f1:
        best_f1, best_threshold = f1, t

print(f"🏆 Best threshold: {best_threshold:.2f}  →  F1 Micro = {best_f1:.4f}")

# ارسم العلاقة
f1_scores = [f1_score(Y_test, (Y_pred_proba >= t).astype(int), average='micro', zero_division=0)
             for t in thresholds]

plt.figure(figsize=(10, 4))
plt.plot(thresholds, f1_scores, 'o-', color='#00d4ff', linewidth=2)
plt.axvline(best_threshold, color='#ff6b6b', linestyle='--', label=f'Best={best_threshold:.2f}')
plt.xlabel('Threshold')
plt.ylabel('F1 Micro')
plt.title('F1 Score vs Classification Threshold', fontsize=13)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 🎬 الخطوة العاشرة: توقع Genre لأي بوستر

In [ ]:
def predict_movie_genre(img_path, threshold=None):
    """توقع genres من صورة بوستر مع عرض بياني للنتائج."""
    if threshold is None:
        threshold = best_threshold

    if not os.path.exists(img_path):
        print(f"❌ الصورة مش موجودة: {img_path}")
        return

    # تحميل وتجهيز الصورة
    img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # توقع
    proba = model.predict(img_array, verbose=0)[0]

    # ترتيب النتائج
    sorted_idx = np.argsort(proba)[::-1]
    top_genres  = [(mlb.classes_[i], proba[i]) for i in sorted_idx[:8]]
    pred_genres = [(g, p) for g, p in top_genres if p >= threshold]

    # عرض
    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(14, 5))

    ax_img.imshow(img)
    ax_img.axis('off')
    ax_img.set_title(os.path.basename(img_path), fontsize=10)

    colors = ['#00d4ff' if p >= threshold else '#444' for _, p in top_genres]
    genres_names = [g for g, _ in top_genres]
    probas = [p for _, p in top_genres]

    bars = ax_bar.barh(genres_names[::-1], probas[::-1], color=colors[::-1])
    ax_bar.axvline(threshold, color='#ff6b6b', linestyle='--', label=f'Threshold={threshold:.2f}')
    ax_bar.set_xlabel('Confidence Score')
    ax_bar.set_title('Genre Predictions', fontsize=13, fontweight='bold')
    ax_bar.legend()
    ax_bar.set_xlim(0, 1)
    for bar, prob in zip(bars, probas[::-1]):
        ax_bar.text(prob + 0.01, bar.get_y() + bar.get_height()/2,
                    f'{prob:.0%}', va='center', fontsize=9)

    plt.tight_layout()
    plt.show()

    print("\n🎯 Predicted Genres:")
    if pred_genres:
        for genre, prob in pred_genres:
            print(f"  ✅ {genre:<20} {prob:.1%}")
    else:
        print("  ⚠️ No genre exceeded threshold. Top prediction:",
              f"{top_genres[0][0]} ({top_genres[0][1]:.1%})")

# ============================================================
# جرب على صورة من الـ test set
# ============================================================
predict_movie_genre(X_test[0])

---
## 💾 الخطوة الأخيرة: حفظ الموديل

In [ ]:
import json

# حفظ الموديل
model.save('final_model_efficientnetB3.keras')
print("✅ Model saved: final_model_efficientnetB3.keras")

# حفظ الـ classes عشان تقدر تستخدمهم لو حملت الموديل تاني
config = {
    'classes':   list(mlb.classes_),
    'threshold': float(best_threshold),
    'img_size':  IMG_SIZE,
    'num_classes': NUM_CLASSES
}
with open('model_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print("✅ Config saved: model_config.json")
print(f"\n🏆 Final Best Threshold: {best_threshold:.2f}")
print(f"🏆 Final F1 Micro:       {best_f1:.4f}")